# 02 · Baselines on clean data

Train on 2013 module runs, test on 2014. A stratified random 80/20 split is shown for comparison only.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

In [2]:
from src.experiments import Data, clean_results
from src.evaluate import MODEL_LABELS
d = Data()
clean, confusions = clean_results(d, n_boot=1000)

In [3]:
cols = ['split', 'model', 'auc', 'auc_lo', 'auc_hi', 'pr_auc', 'recall', 'precision', 'f1', 'false_alarm_rate']
clean[cols].round(3)

,split,model,auc,auc_lo,auc_hi,pr_auc,recall,precision,f1,false_alarm_rate
0,time (2013 -> 2014),baseline,0.500,0.500,0.500,0.439,0.000,0.000,0.000,0.000
1,time (2013 -> 2014),logreg,0.718,0.710,0.725,0.673,0.571,0.631,0.600,0.261
2,time (2013 -> 2014),hgb,0.747,0.739,0.755,0.704,0.601,0.660,0.629,0.242
3,random 80/20,baseline,0.500,0.500,0.500,0.441,0.000,0.000,0.000,0.000
4,random 80/20,logreg,0.732,0.719,0.745,0.689,0.662,0.619,0.640,0.321
5,random 80/20,hgb,0.775,0.763,0.787,0.750,0.606,0.702,0.650,0.203


Confusion matrices on the 2014 test set, threshold 0.5:

In [4]:
for name, cm in confusions.items():
    print(MODEL_LABELS[name]); print(cm, end='\n\n')

Majority class
                   pred: not at risk  pred: at risk
true: not at risk               8813              0
true: at risk                   6907              0

Logistic regression
                   pred: not at risk  pred: at risk
true: not at risk               6510           2303
true: at risk                   2963           3944

Gradient boosting
                   pred: not at risk  pred: at risk
true: not at risk               6676           2137
true: at risk                   2753           4154



## Why the time split is the honest one

The random split mixes 2013 and 2014 students, so the model has seen the same module runs, the same course design and the same cohort it is tested on. In real use the model is trained on last year and applied to this year, where course materials, cohorts and module offerings change (module CCC only runs in 2014, so the time split also tests on a module the model never saw). The gap between the two rows above is how much a random split flatters the model.

## What the gradient-boosting model relies on

Permutation importance on the 2014 test set (drop in AUC when one feature is shuffled):

In [5]:
from sklearn.inspection import permutation_importance
from src.evaluate import make_model
Xtr, ytr = d.X.loc[d.train_sids], d.y.loc[d.train_sids]
Xte, yte = d.X.loc[d.test_sids], d.y.loc[d.test_sids]
hgb = make_model('hgb', Xtr).fit(Xtr, ytr)
imp = permutation_importance(hgb, Xte, yte, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=Xte.columns).sort_values(ascending=False).round(4).head(12)

mean_score              0.1161
studied_credits         0.0265
highest_education       0.0127
active_days             0.0127
n_late                  0.0125
days_since_last         0.0081
clicks_quiz             0.0077
clicks_oucontent        0.0076
imd_band                0.0066
clicks_w2               0.0064
num_of_prev_attempts    0.0049
n_submitted             0.0039
dtype: float64